<a href="https://colab.research.google.com/github/Noadori34/Project-data-science/blob/main/notebooks/04_validated_cross_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook follows DS_28 notebook and our objective is to reproduce the full-image versus kidney-focused EfficientNet comparison using a more reliable evaluation protocol.

In [1]:
# Initializing and imports:


import os
import random
import subprocess
import numpy as np
import pandas as pd
import torch

REPO_DIR = "/content/Project-data-science"
REPO_URL = "https://github.com/Noadori34/Project-data-science.git"

if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        check=True
    )

os.chdir(REPO_DIR)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)
print("Repository:", os.getcwd())


Device: cuda
Repository: /content/Project-data-science


In [2]:
# Load and verify existing folds:


from pathlib import Path

folds_path = Path("results/eda/folds.csv")
pairs_path = Path("results/eda/duplicate_pairs.csv")

folds_df = pd.read_csv(folds_path)
pairs_df = pd.read_csv(pairs_path)

print("Total images:", len(folds_df))
print("Unique filenames:", folds_df["filename"].nunique())
print("Number of folds:", folds_df["fold"].nunique())

print("\nClass distribution:")
print(pd.crosstab(
    folds_df["fold"],
    folds_df["diagnosis"]
))

# Verify that each declared group belongs to one fold
group_folds = folds_df.groupby("group_id")["fold"].nunique()

print("\nGroups crossing folds:", (group_folds > 1).sum())

# Inspect the identified problematic pair
print("\nImages 541 and 542:")
display(
    folds_df[
        folds_df["filename"].isin(["541.jpg", "542.jpg"])
    ]
)


Total images: 1985
Unique filenames: 1985
Number of folds: 5

Class distribution:
diagnosis   0    1
fold              
0          90  307
1          90  307
2          90  306
3          90  306
4          90  309

Groups crossing folds: 0

Images 541 and 542:


,filename,diagnosis,group_id,fold
1477,541.jpg,1,group_65,1
1478,542.jpg,1,group_66,2


We already know that 541.jpg and 542.jpg were separated across two folds. The next operation should be to merge their similarity groups and regenerate the splits using StratifiedGroupKFold.

In [3]:
# Correct duplicate groups and regenerate folds


from sklearn.model_selection import StratifiedGroupKFold

# Preserve original assignments
corrected_df = folds_df.copy()

# Union-find groups connected images together
parent = {
    filename: filename
    for filename in corrected_df["filename"]
}

def find(x):
    if parent[x] != x:
        parent[x] = find(parent[x])
    return parent[x]

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

# Preserve ALL existing grouping relationships
for _, group in corrected_df.groupby("group_id"):
    filenames = group["filename"].tolist()
    for filename in filenames[1:]:
        union(filenames[0], filename)

# Merge images classified as identical or same examination
for _, pair in pairs_df.iterrows():
    verdict = str(pair["verdict"]).lower()

    if (
        verdict == "identical image"
        or verdict.startswith("same exam")
    ):
        a = f"{int(pair['img_a'])}.jpg"
        b = f"{int(pair['img_b'])}.jpg"

        if a in parent and b in parent:
            union(a, b)

# Assign corrected group IDs
corrected_df["corrected_group"] = (
    corrected_df["filename"].map(find)
)

print("Original groups:", corrected_df["group_id"].nunique())
print("Corrected groups:", corrected_df["corrected_group"].nunique())

display(
    corrected_df[
        corrected_df["filename"].isin(
            ["541.jpg", "542.jpg", "4.jpg", "296.jpg", "297.jpg"]
        )
    ][["filename", "diagnosis", "corrected_group"]]
)


Original groups: 1893
Corrected groups: 1892


,filename,diagnosis,corrected_group
1204,296.jpg,0,296.jpg
1205,297.jpg,1,296.jpg
1319,4.jpg,1,296.jpg
1477,541.jpg,1,541.jpg
1478,542.jpg,1,541.jpg


In [4]:
# Generate new group folds


sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

corrected_df["fold"] = -1

for fold, (_, test_idx) in enumerate(
    sgkf.split(
        X=corrected_df["filename"],
        y=corrected_df["diagnosis"],
        groups=corrected_df["corrected_group"]
    )
):
    corrected_df.loc[test_idx, "fold"] = fold

# Verify every image is assigned
assert (corrected_df["fold"] >= 0).all()

# Verify groups do not cross folds
group_check = (
    corrected_df.groupby("corrected_group")["fold"].nunique()
)

assert (group_check == 1).all(), \
    "Some related image groups cross folds!"

# Verify identified related pairs
for _, pair in pairs_df.iterrows():
    verdict = str(pair["verdict"]).lower()

    if verdict == "identical image" or verdict.startswith("same exam"):
        a = f"{int(pair['img_a'])}.jpg"
        b = f"{int(pair['img_b'])}.jpg"

        fa = corrected_df.loc[
            corrected_df["filename"] == a, "fold"
        ].iloc[0]

        fb = corrected_df.loc[
            corrected_df["filename"] == b, "fold"
        ].iloc[0]

        assert fa == fb, f"Cross-fold pair found: {a}, {b}"

print("Corrected fold distribution:")
print(pd.crosstab(
    corrected_df["fold"],
    corrected_df["diagnosis"]
))

print("\nGroups crossing folds:",
      (group_check > 1).sum())


Corrected fold distribution:
diagnosis   0    1
fold              
0          91  312
1          97  291
2          96  304
3          78  323
4          88  305

Groups crossing folds: 0


In [5]:
# Save corrected folds


output_path = Path("results/eda/folds_corrected.csv")

export_df = corrected_df[
    ["filename", "diagnosis", "corrected_group", "fold"]
].rename(columns={
    "corrected_group": "group_id"
})

export_df.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Images:", len(export_df))
print("Groups:", export_df["group_id"].nunique())


Saved: results/eda/folds_corrected.csv
Images: 1985
Groups: 1892


In [6]:
# Generate inner training and validation assignments


from sklearn.model_selection import StratifiedGroupKFold
import pandas as pd

# Use the corrected fold assignments
data_df = export_df.copy()

# Store the inner splits for reproducibility
inner_split_records = []

for outer_fold in range(5):

    # Reserve one outer fold for independent evaluation
    development_df = data_df[
        data_df["fold"] != outer_fold
    ].copy()

    outer_test_df = data_df[
        data_df["fold"] == outer_fold
    ].copy()

    # Create a group-aware inner validation split
    inner_sgkf = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=42 + outer_fold
    )

    train_idx, val_idx = next(
        inner_sgkf.split(
            development_df["filename"],
            development_df["diagnosis"],
            development_df["group_id"]
        )
    )

    train_df = development_df.iloc[train_idx]
    val_df = development_df.iloc[val_idx]

    # Check group separation
    train_groups = set(train_df["group_id"])
    val_groups = set(val_df["group_id"])
    test_groups = set(outer_test_df["group_id"])

    assert train_groups.isdisjoint(val_groups)
    assert train_groups.isdisjoint(test_groups)
    assert val_groups.isdisjoint(test_groups)

    print(
        f"Outer fold {outer_fold}: "
        f"Train={len(train_df)}, "
        f"Validation={len(val_df)}, "
        f"Test={len(outer_test_df)}"
    )

    for split_name, split_df in [
        ("train", train_df),
        ("validation", val_df),
        ("test", outer_test_df)
    ]:
        for _, row in split_df.iterrows():
            inner_split_records.append({
                "outer_fold": outer_fold,
                "filename": row["filename"],
                "diagnosis": row["diagnosis"],
                "group_id": row["group_id"],
                "split": split_name
            })

inner_splits_df = pd.DataFrame(inner_split_records)

inner_splits_df.to_csv(
    "results/eda/nested_cv_splits.csv",
    index=False
)

print("\nSaved nested cross-validation assignments.")


Outer fold 0: Train=1260, Validation=322, Test=403
Outer fold 1: Train=1277, Validation=320, Test=388
Outer fold 2: Train=1267, Validation=318, Test=400
Outer fold 3: Train=1270, Validation=314, Test=401
Outer fold 4: Train=1271, Validation=321, Test=393

Saved nested cross-validation assignments.


In [7]:
# final split and integrity tests


for outer_fold in range(5):
    subset = inner_splits_df[
        inner_splits_df["outer_fold"] == outer_fold
    ]

    assert len(subset) == 1985
    assert subset["filename"].nunique() == 1985

    print(f"\nOUTER FOLD {outer_fold}")
    print(pd.crosstab(
        subset["split"],
        subset["diagnosis"]
    ))

    print("\nPathological proportion:")
    print(
        subset.groupby("split")["diagnosis"]
        .mean()
        .round(3)
    )

print("\nAll outer iterations contain each image exactly once.")



OUTER FOLD 0
diagnosis     0    1
split               
test         91  312
train       286  974
validation   73  249

Pathological proportion:
split
test          0.774
train         0.773
validation    0.773
Name: diagnosis, dtype: float64

OUTER FOLD 1
diagnosis     0    1
split               
test         97  291
train       283  994
validation   70  250

Pathological proportion:
split
test          0.750
train         0.778
validation    0.781
Name: diagnosis, dtype: float64

OUTER FOLD 2
diagnosis     0    1
split               
test         96  304
train       291  976
validation   63  255

Pathological proportion:
split
test          0.760
train         0.770
validation    0.802
Name: diagnosis, dtype: float64

OUTER FOLD 3
diagnosis     0    1
split               
test         78  323
train       296  974
validation   76  238

Pathological proportion:
split
test          0.805
train         0.767
validation    0.758
Name: diagnosis, dtype: float64

OUTER FOLD 4
diagnosis     

In [8]:
# Verify image masks paths


from pathlib import Path

IMAGES_DIR = Path("bbdd_kidney/images")
LABELS_DIR = Path("bbdd_kidney/labels")
MASKS_DIR = Path("bbdd_kidney/masks")

# Confirm expected dataset structure
assert IMAGES_DIR.is_dir(), "Image folder not found"
assert LABELS_DIR.is_dir(), "Labels folder not found"
assert MASKS_DIR.is_dir(), "Masks folder not found"

# Verify every filename in our corrected split
missing_images = []
missing_masks = []

for filename in export_df["filename"]:
    image_path = IMAGES_DIR / filename
    mask_path = MASKS_DIR / f"{Path(filename).stem}.mat"

    if not image_path.is_file():
        missing_images.append(filename)

    if not mask_path.is_file():
        missing_masks.append(filename)

print("Missing images:", len(missing_images))
print("Missing masks:", len(missing_masks))

assert not missing_images, missing_images[:10]
assert not missing_masks, missing_masks[:10]

print("Dataset paths verified successfully.")


Missing images: 0
Missing masks: 0
Dataset paths verified successfully.
